# Variant Prioritization and Interpretation

**Estimated time:** 25 minutes

Combine the paper, GTEx, HuBMAP, and Pharos results without treating them as
equivalent. Then use the combined table to choose a follow-up question.


## Combine the results

The paper reports 54 variants across 25 genes. The analysis adds information
from GTEx, HuBMAP, and Pharos to each variant based on its gene. This
information can help prioritize variants for follow-up, but it does not change
the classifications reported in the paper.


### Keep evidence types separate

Each source measures a different level of biology, even when the results appear
in the same table:

| Source | What it provides | How it informs follow-up |
|---|---|---|
| Paper fields | The exact variant, phenotype, score, and class reported by the study | Which reported variant and phenotype are being evaluated? |
| GTEx | Median gene expression reported in the selected reference heart tissues | Which heart tissue is appropriate for follow-up? |
| HuBMAP | Whether indexed values are available in up to the first 500 ventricular cardiac-myocyte records | Are these cells suitable for follow-up, or is another atlas or experiment needed? |
| Pharos | The current protein annotations and target development level | Should follow-up begin with a known drug relationship, chemical probe, biological mechanism, or basic protein characterization? |

Combining these fields into one pathogenicity score would hide these
differences. We will use each column only for the question it can answer.

The gene with the largest value or most developed target is not automatically
the best candidate genetic variant. The goal is a justified next step with
each result tied to its source.


## Build the combined table


### Load the source data and API helpers

Load the 54 published variant rows and import the GTEx, HuBMAP, and Pharos
wrappers.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import requests

# Locate the repository root.
REPO_ROOT = Path.cwd() if Path("api_helpers.py").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Import the API wrappers.
from api_helpers import (
    fetch_gtex_context,
    fetch_hubmap_ventricular_context,
    fetch_pharos_context,
)

# Load the published variants.
DATA_DIR = REPO_ROOT / "data"
variants = pd.read_csv(DATA_DIR / "variants.csv")
variants.head()


One row represents one published observation with its gene, HGVS descriptions,
study class, phenotype, and source. Each join must preserve all 54 rows.


### Request gene-level data

Query all three APIs for the 25 genes. HuBMAP takes the longest because it checks
each gene separately.


In [ ]:
# Keep one copy of each gene symbol.
gene_symbols = sorted(variants["gene_symbol"].unique())
gtex = fetch_gtex_context(gene_symbols)
hubmap = fetch_hubmap_ventricular_context(gene_symbols)
pharos = fetch_pharos_context(gene_symbols)

retrieval_summary = pd.DataFrame(
    {
        "resource": ["GTEx", "HuBMAP", "Pharos"],
        "rows": [len(gtex), len(hubmap), len(pharos)],
    }
)
retrieval_summary


The dated teaching responses return 50 GTEx rows, one for each of 25 genes in
two tissues, along with 25 HuBMAP rows and 25 Pharos rows. At this stage, every
table describes genes rather than individual variants.


### Build one row per gene

Create one row per gene with two GTEx tissues, one HuBMAP cell type, and selected
Pharos fields. This structure prevents extra variant rows during the final join.


In [ ]:
# Reshape GTEx tissues into columns.
gtex_wide = gtex.pivot(
    index="gene_symbol",
    columns="tissue_id",
    values="median_tpm",
).rename(
    columns={
        "Heart_Atrial_Appendage": "gtex_atrial_tpm",
        "Heart_Left_Ventricle": "gtex_ventricle_tpm",
    }
)

# Select ventricular cardiac myocytes.
ventricular = (
    hubmap[hubmap["cell_type_id"] == "CL:0002131"]
    .loc[
        :,
        [
            "gene_symbol",
            "mean_normalized_expression",
            "percent_detected",
            "availability",
        ],
    ]
    .rename(
        columns={
            "mean_normalized_expression": "hubmap_ventricular_mean",
            "percent_detected": "hubmap_ventricular_percent_detected",
            "availability": "hubmap_availability",
        }
    )
)

# Join one-to-one gene records.
gene_results = (
    gtex_wide.reset_index()
    .merge(ventricular, on="gene_symbol", how="left", validate="one_to_one")
    .merge(
        pharos.loc[:, ["gene_symbol", "tdl", "drug_count"]],
        on="gene_symbol",
        how="left",
        validate="one_to_one",
    )
)

# Confirm that all 25 genes remain.
assert len(gene_results) == 25
gene_results.head()


The 25-row `gene_results` table contains GTEx median TPM, HuBMAP expression and
availability, and Pharos `tdl` and `drug_count`. Column prefixes identify each
source.


### Join gene-level data to every variant

Run the next cell to match the 25 gene rows back to all 54 variant rows while
keeping every row from the published variant table.


In [ ]:
# Join gene-level data by gene symbol.
combined_variants = variants.merge(
    gene_results,
    on="gene_symbol",
    how="left",
    validate="many_to_one",
)

# Confirm that all 54 variant rows remain.
assert len(combined_variants) == len(variants) == 54

# Inspect variant and gene-level fields together.
combined_variants.loc[
    :,
    [
        "subject_id",
        "gene_symbol",
        "hgvs_c",
        "study_class",
        "phenotype",
        "gtex_ventricle_tpm",
        "hubmap_availability",
        "tdl",
    ],
].head(10)


The combined table places each variant beside the corresponding GTEx, HuBMAP, and Pharos
fields. A many-to-one join allows several variants to share gene data while
preserving all 54 published rows.


## Prioritize a focused DCM follow-up set

We focus on dilated cardiomyopathy, abbreviated DCM, and ventricular cardiac
myocytes.


### Define the follow-up question

Which DCM variant rows have HuBMAP values for ventricular cardiac myocytes, and
how do they rank by GTEx left-ventricle expression?


In [ ]:
# Select DCM rows with measured HuBMAP values.
dcm_follow_up = (
    combined_variants[
        (combined_variants["phenotype"] == "DCM")
        & (combined_variants["hubmap_availability"] == "available")
    ]
    .sort_values(
        ["gtex_ventricle_tpm", "gene_symbol", "subject_id"],
        ascending=[False, True, True],
    )
    .loc[
        :,
        [
            "subject_id",
            "gene_symbol",
            "hgvs_c",
            "study_class",
            "gtex_ventricle_tpm",
            "hubmap_ventricular_percent_detected",
            "tdl",
        ],
    ]
)

# Inspect the combined evidence.
dcm_follow_up


How many rows remain? Which genes contribute P or LP findings? Which VUS occur
in those genes?

The filter returns 15 of the 28 DCM rows. Ten are P or LP in the paper, and five
are VUS. It excludes the other 13 only because the selected HuBMAP index did
not return a ventricular cardiac-myocyte value. This filter does not rank their
clinical importance.


## Examine one missense variant with ProtVar

The paper classified *TNNT2* `c.776A>C` (p.Asp259Ala) as a VUS. *TNNT2*
encodes cardiac troponin T, a component of the troponin complex that helps
regulate calcium-dependent contraction in heart muscle. Two other participants
with DCM carried pathogenic or likely pathogenic *TNNT2* variants. These
findings support the relevance of *TNNT2* to DCM in this cohort, but they do not
determine the effect of p.Asp259Ala.

ProtVar provides variant-level predictions for p.Asp259Ala. AlphaMissense predicts
that the substitution may affect protein function, EVE returns an uncertain
prediction, and FoldX does not predict a large change in overall protein
stability.

The AlphaFold pLDDT score reports confidence in the local structure around
residue 259. It is not a prediction of the variant's effect.

Learn more about the returned protein scores in the
[ProtVar API documentation](https://www.ebi.ac.uk/ProtVar/api/swagger-ui/index.html).


### Request protein predictions

ProtVar maps p.Asp259Ala to UniProt accession P45379 at residue 259. Query the
score endpoint for AlphaMissense and EVE, then query the FoldX stability
endpoint.


In [ ]:
protvar_api = "https://www.ebi.ac.uk/ProtVar/api"
protein_accession = "P45379"
protein_position = 259
alternate_amino_acid = "A"

# Compare two variant-effect predictions for the same substitution.
alphamissense_response = requests.get(
    f"{protvar_api}/score/{protein_accession}/{protein_position}",
    params={"mt": alternate_amino_acid, "type": "AM"},
    timeout=30,
)
alphamissense_response.raise_for_status()
alphamissense = alphamissense_response.json()[0]

eve_response = requests.get(
    f"{protvar_api}/score/{protein_accession}/{protein_position}",
    params={"mt": alternate_amino_acid, "type": "EVE"},
    timeout=30,
)
eve_response.raise_for_status()
eve = eve_response.json()[0]

# Ask how the substitution is predicted to change protein stability.
foldx_response = requests.get(
    f"{protvar_api}/prediction/foldx/{protein_accession}/{protein_position}",
    params={"variantAA": alternate_amino_acid},
    timeout=30,
)
foldx_response.raise_for_status()
foldx = foldx_response.json()[0]

# Display three results that help plan a follow-up experiment.
protvar_summary = pd.DataFrame(
    {
        "prediction": ["AlphaMissense", "EVE", "FoldX stability"],
        "result": [
            (
                f"{alphamissense['amClass'].title()} "
                f"({alphamissense['amPathogenicity']:.4f})"
            ),
            (
                f"{eve['eveClass'].title()} "
                f"({eve['score']:.3f})"
            ),
            (
                f"{foldx['foldxDdg']:.3f} kcal/mol; "
                f"AlphaFold pLDDT {foldx['plddt']:.2f}"
            ),
        ],
    }
)

protvar_summary


GTEx shows high *TNNT2* expression in heart tissue, and HuBMAP reports expression
in ventricular cardiac myocytes. Pharos identifies cardiac troponin T as a
biologically characterized protein with extensive interaction records. Cardiac
troponin T interacts with troponin I, troponin C, and tropomyosin to regulate
thin-filament contraction.

The next step would be to compare wild-type and D259A troponin complexes across
calcium concentrations to test whether the substitution changes
calcium-regulated thin-filament activity. The study classification remains VUS.


## Interpret the integrated evidence

Interpret each result according to what its source measures.


### Interpret the DCM follow-up set

The paper's classification remains attached to each variant. The API results
help select genes and experimental systems for follow-up.
[Birch and colleagues](https://doi.org/10.1186/s12967-025-07586-w) use a similar
approach by combining variant classification, phenotype match, and molecular
results while reporting the strength of each finding separately.

The label **P/LP with cardiac expression support** means that the paper
classified a variant as pathogenic or likely pathogenic and both GTEx and
HuBMAP returned relevant cardiac gene-expression values. This combination
supports follow-up but does not prove the variant's mechanism.

| Our interpretation | Variants in `dcm_follow_up` | What the evidence supports |
|---|---|---|
| P/LP with cardiac expression support | *DES* `c.735G>A`; *ACTC1* `c.301G>A`; *TNNT2* `c.547C>T`; *TNNT2* `c.547C>G`; *MYBPC3* `c.2490dup` in two subjects; *MYBPC3* `c.442G>A`; *PLN* `c.25C>T`; *MYLK3* `c.618dup`; *MYLK3* `c.1569-2A>C` | The paper classified these variants as P/LP, and GTEx and HuBMAP support cardiac follow-up for their genes. |
| VUS with cardiac expression support | *TNNI3* `c.337G>A`; *ACTC1* `c.1132T>C`; *TNNT2* `c.776A>C`; *FLNC* `c.4181A>G`; *TNNI3K* `c.827+1G>T` | These remain VUS. Cardiac expression supports studying the genes but does not establish the effect of an allele. |

**What the integrated evidence added:**
Variant analysis often leaves several credible candidate genetic variants. The
next decision is which candidate genetic variant to investigate first and which
experimental system fits the question.

All 25 genes had measurable expression in the selected GTEx heart tissues, so
GTEx did not substantially narrow the list. HuBMAP coverage provided the more
selective filter.

Within this analysis, the results for *DES*, *TNNT2*, *MYBPC3*, *ACTC1*, *PLN*,
and *MYLK3* support using cardiac myocytes to study the reported variants. The
results also identify possible experimental tools and follow-up questions.


### Compare variants within the same gene

All three *TNNT2* rows receive the same GTEx, HuBMAP, and Pharos values. Yet the
paper classified `c.547C>T` as P, `c.547C>G` as LP, and `c.776A>C` as VUS.
*TNNT2* had a GTEx left-ventricle median of 2,896.66 TPM and a value above zero
in 38.2% of retrieved HuBMAP ventricular cardiac-myocyte records. These results
support a cardiac-cell model, but only variant-specific testing can distinguish
the alleles' effects.


### Identify possible experimental starting points

The paper describes *MYLK3* as an emerging DCM gene and classified both variants
as LP. *MYLK3* had a GTEx left-ventricle median of 39.99 TPM and a value above
zero in 13.4% of retrieved HuBMAP ventricular cardiac-myocyte records. Pharos
classifies the protein as `Tchem` and reports five ligands and one drug
relationship. These records identify compounds to inspect before designing
functional studies of *MYLK3*.

Pharos classifies *TNNI3K* as `Tchem`, but *TNNI3K* had a value above zero in
only 0.4% of retrieved ventricular cardiac-myocyte records. Its chemical
records may still be useful, but these HuBMAP results provide less support for
it over the P/LP findings above.


## Keep missing and repeated observations visible

Coverage gaps and repeated variants affect which findings are prioritized.


### Keep coverage gaps visible

Create a unique gene list of HuBMAP coverage gaps. These genes may require
another atlas, another cell type, or direct measurement.


In [ ]:
# Select HuBMAP coverage gaps.
coverage_gaps = (
    combined_variants[
        combined_variants["hubmap_availability"]
        != "available"
    ]
    .loc[:, ["gene_symbol", "hubmap_availability"]]
    .drop_duplicates()
    .sort_values("gene_symbol")
    .reset_index(drop=True)
)

# Keep one row per unavailable gene.
coverage_gaps.head()


The 13 DCM rows absent from `dcm_follow_up` remain in the full table. They
include P/LP findings in *MYL3*, *TTN*, *GYG1*, *LMNA*, and *DMD*, plus one
*LMNA* VUS. HuBMAP did not return values for the selected cell type. Studying
cell-level expression for these genes would require another atlas, cell type,
or direct measurement.


### Identify recurrent variants

Find exact variant observations that occur in more than one participant. The
grouping uses both coding and protein HGVS fields so distinct variants are not
combined accidentally.


In [ ]:
# Count exact reported variants across participants.
recurrent_variants = (
    variants.groupby(
        ["gene_symbol", "hgvs_c", "hgvs_p"],
        dropna=False,
    )
    .agg(
        subjects=("subject_id", lambda values: ", ".join(map(str, values))),
        observations=("subject_id", "size"),
    )
    .reset_index()
    .query("observations > 1")
    .sort_values(["observations", "gene_symbol"], ascending=[False, True])
)

# Show recurrent observations.
recurrent_variants.head()


Three variants appear in more than one participant. *MYBPC3* `c.2490dup`
appears in three participants, while *LMNA* `c.1304_1307dup` and *TTR*
`c.323A>G` each appear in two. Recurrence describes this study cohort. It does
not establish pathogenicity or population frequency.


## Quiz yourself!
How many rows should remain after gene-level data are joined back to the
variants?

- 54
- 25
- 46

<details>
<summary>Show answer and feedback</summary>

- **54:** Correct. The join returns the 25 gene-level data rows to the full variant table without removing published variant records.
- **25:** This is the number of gene-level data rows before they are joined back to the variants.
- **46:** This is the number of represented subjects, not the expected join size.

</details>

Why is this a many-to-one join?

- Several variants can share one gene
- Every variant has several GTEx tissues

<details>
<summary>Show answer and feedback</summary>

- **Several variants can share one gene:** Correct. The table has 54 variant rows but only 25 genes, so one gene-level data row may match multiple variants.
- **Every variant has several GTEx tissues:** The relationship refers to multiple variant rows matching one gene-level data row, not to the number of GTEx tissues.

</details>


## Optional activity: Interpret one candidate genetic variant

Practice separating variant-level and gene-level evidence.

Choose one row from `dcm_follow_up` and write three sentences:

1. Report the exact variant, phenotype, and study class.
2. Describe its GTEx and HuBMAP results with one limitation.
3. Describe its Pharos protein information and propose one experiment or
   additional data source for follow-up.

The paper supplies the variant class. The APIs supply gene and protein evidence
for follow-up.


## Key points

- The many-to-one join preserves all 54 published variant observations.
- In the dated teaching data, 15 DCM rows have indexed ventricular
  cardiac-myocyte values; 10 are P or LP in the paper and 5 are VUS.
- Tissue expression, cell-type measurements, and protein information can guide
  selection of candidate genetic variants and models without changing study
  classifications.
- Comparing AlphaMissense, EVE, and FoldX for one VUS shows how variant-level
  predictions can refine the next experiment without replacing the study class.

**Next:** Review the workflow and apply it to another expert-reviewed variant
table.
